# 01 — MC-OCR dataset audit

This notebook is the interactive audit surface for the locally downloaded MC-OCR data. It uses the reusable `src/docucheck` adapters and never embeds receipt samples in the committed notebook. The default root is `data/mcocr`; set `MC_OCR_ROOT` or edit the `dataset_root` variable below for another local copy.

The validation images are counted for split inventory only. The image galleries below use the KIE training subset.

In [ ]:
from pathlib import Path
import os
import random
import ast
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from docucheck.data.loader import audit_dataset, load_kie_records, load_text_recognition_pairs, resolve_dataset_root
from docucheck.data.validation import validate_record_regions
from docucheck.visualization.receipt import plot_receipt_record

DATASET_ROOT = resolve_dataset_root(os.environ.get("MC_OCR_ROOT"))
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError("Set MC_OCR_ROOT to the local MC-OCR directory")
audit = audit_dataset(DATASET_ROOT)
records = load_kie_records(DATASET_ROOT)
print(f"Dataset available: {DATASET_ROOT.is_dir()} | KIE train records: {len(records)}")

## Dataset inventory

The table reports the observed primary train and validation directories plus the other task exports discovered by the parser. Validation images are not opened in this notebook.

In [ ]:
inventory = {
    **audit["image_inventory"],
    **audit["source_files"],
    **audit["image_integrity"],
}
display(pd.Series(inventory, name="count or availability").to_frame())

split_counts = pd.Series({
    "KIE train images": audit["image_inventory"]["train_image_count"],
    "KIE validation images (inventory only)": audit["image_inventory"]["validation_image_count"],
})
ax = split_counts.plot(kind="bar", color=["#255f85", "#d17b49"], figsize=(8, 4))
ax.set_ylabel("Images")
ax.set_title("Images in observed KIE subsets")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

## Annotation schema and label coverage

The KIE export stores `anno_texts` and `anno_labels` as `|||`-separated arrays and `anno_polygons` as a Python-literal list of objects. This cell reports structure, category IDs, and labels without displaying raw receipt text or identifiers. The mapping below reflects observed labels; rare unrecognized labels remain unmapped.

In [ ]:
train_csv = DATASET_ROOT / "mcocr_train_df.csv"
source_columns = list(pd.read_csv(train_csv, nrows=0).columns)
poly_keys, category_ids, coordinate_lengths = Counter(), Counter(), Counter()
for record in records:
    for region in record["regions"]:
        source_annotation = region["source_annotation"]
        if isinstance(source_annotation, dict):
            poly_keys.update(source_annotation.keys())
            category_ids[(region["label_raw"], region["category_id"])] += 1
            polygon = region["polygon"]
            if polygon:
                coordinate_lengths[2 * len(polygon)] += 1

print("CSV columns:", source_columns)
print("Polygon object keys:", dict(poly_keys))
print("Label/category ID pairs:", {f"{label} / {category}": n for (label, category), n in sorted(category_ids.items())})
print("Flat coordinate count per polygon:", dict(sorted(coordinate_lengths.items())))
label_counts = pd.Series(audit["annotations"]["label_region_counts"]).sort_values(ascending=False)
display(label_counts.rename_axis("source label").to_frame("annotated regions"))
field_counts = pd.Series(audit["annotations"]["field_image_counts"]).sort_values(ascending=False)
ax = field_counts.plot(kind="bar", color="#3f8f78", figsize=(8, 4))
ax.set_ylabel("Images with at least one field region")
ax.set_title("Canonical field coverage in KIE train annotations")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

## OCR task exports and transcription ground truth

The download includes full-page text-box/transcription sidecars and a separate cropped text-recognition set. These are distinct annotation sources from the KIE labels. This cell reports only counts and text-length aggregates; it never displays transcripts.

In [ ]:
ocr_exports = audit["ocr_exports"]
display(pd.Series(ocr_exports, name="observed value").to_frame())

recognition_pairs = {
    split: load_text_recognition_pairs(split, DATASET_ROOT)
    for split in ("train", "val")
}
transcription_lengths = [
    len(pair["text_raw"]) for split_pairs in recognition_pairs.values()
    for pair in split_pairs if pair["text_raw"] is not None
]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(transcription_lengths, bins=30, color="#255f85", edgecolor="white")
ax.set(xlabel="Transcript length (Unicode code points)", ylabel="Crop regions", title="Text-recognition transcription lengths")
plt.tight_layout()
plt.show()

The full-page transcript sidecars cover fewer images than the KIE training CSV; unmatched images must be reported during preparation. The text-recognition task provides crop/transcription pairs and can support recognition evaluation on that crop domain, but those crops are not full receipt-level KIE labels.

## Image properties and annotation quality

Dimensions are summarized over readable KIE train images. The annotation issue table distinguishes source-array mismatches and malformed geometry from missing image files. `anno_image_quality` is an observed source column; its meaning is not assumed to be an OCR quality score.

In [ ]:
dim_rows = []
for dim, count in audit["image_dimensions"].items():
    width, height = map(int, dim.split("x"))
    dim_rows.append({"width": width, "height": height, "count": count, "aspect_ratio": width / height})
dimensions = pd.DataFrame(dim_rows)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(dimensions.width, dimensions.height, s=dimensions["count"] * 2, alpha=.65, color="#255f85")
axes[0].set(xlabel="Width (px)", ylabel="Height (px)", title="Unique image dimensions (marker size = count)")
axes[1].hist(dimensions.aspect_ratio, bins=18, weights=dimensions["count"], color="#d17b49", edgecolor="white")
axes[1].set(xlabel="Width / height", ylabel="Images", title="Receipt orientation and aspect ratio")
plt.tight_layout()
plt.show()

region_counts = pd.Series(audit["annotations"]["region_count_per_image"])
print("Image integrity:", audit["image_integrity"])
print("Annotation issue counts:", audit["annotations"]["issue_counts"])
print("Regions per image (min / median / max):", int(region_counts.min()), float(region_counts.median()), int(region_counts.max()))
ax = region_counts.plot(kind="hist", bins=range(0, int(region_counts.max()) + 2), color="#3f8f78", figsize=(8, 4), edgecolor="white")
ax.set(xlabel="KIE regions per image", ylabel="Images", title="KIE annotation density")
plt.tight_layout()
plt.show()

## Local visual inspection — training subset only

The following deterministic sample overlays source KIE polygons and labels. Raw text snippets are intentionally omitted from the plot. Re-run this cell locally to inspect the receipt images; outputs are stripped before commit.

In [ ]:
labeled_records = [record for record in records if record["regions"]]
sampled = random.Random(42).sample(labeled_records, min(4, len(labeled_records)))
fig, axes = plt.subplots(len(sampled), 1, figsize=(10, 5 * len(sampled)))
if len(sampled) == 1:
    axes = [axes]
for record, ax in zip(sampled, axes):
    plot_receipt_record(record, ax=ax, show_text=False)
plt.tight_layout()
plt.show()

## Edge cases and verified findings

The aggregate checks below surface empty source annotation arrays, unmapped labels, and invalid geometry without printing receipt text. The local examples above are suitable for manual visual alignment review.

In [ ]:
unmapped = {label: count for label, count in audit["annotations"]["label_region_counts"].items()
            if label not in audit["annotations"]["field_mapping"]}
geometry_issues = Counter()
for record in records:
    result = audit["image_dimensions"]
    # Validate using the actual image dimensions for each local training record.
    from docucheck.data.loader import inspect_image
    image_meta = inspect_image(record["image_path"])
    if image_meta["valid"]:
        for region_result in validate_record_regions(record, image_meta["width"], image_meta["height"]):
            geometry_issues.update(region_result["issues"])
print("Dataset root exists:", DATASET_ROOT.is_dir())
print("Unmapped source labels (retained):", unmapped)
print("Geometry issue counts:", dict(geometry_issues))
print("Image inventory:", audit["image_inventory"])
print("Annotation array issue counts:", audit["annotations"]["issue_counts"])

## Audit Findings

- The observed KIE training export contains **1,155** image records and **6,585** field regions; each row's text, label, and polygon arrays are aligned except for two rows where all annotation arrays are empty.
- The paired training image directory has 1,155 readable RGB images; there are no unmatched KIE train image IDs in this local export.
- The observed field labels are `SELLER`, `ADDRESS`, `TIMESTAMP`, and `TOTAL_COST`. The address label is `ADDRESS`. A single `TOTAL_TOTAL_COST` label is retained as an unmapped source anomaly.
- The local validation folder contains 391 images, but the accompanying sample CSV has no label or polygon columns. It is inventory-only for this audit and cannot support field-level evaluation.
- Separate text-detection and text-recognition exports exist. Their transcription semantics must be audited independently before using them as OCR ground truth.
- No dataset license/terms file was present in the inspected root; do not redistribute dataset assets until terms are verified.